# 05 — SHAP Explainability Analysis

**Why this matters:** The supervisor grades explainability heavily —  
"the best model is evaluated not only by performance but *to a big part* by explainability using SHAP values."

## What this notebook covers

| Analysis | Models | Output |
|----------|--------|--------|
| SHAP beeswarm (importance + direction) | All 7 | `shap/shap_summary_{model}.png` |
| SHAP bar (mean \|SHAP\|) | All 7 | `shap/shap_importance_{model}.png` |
| SHAP dependence plots (top-3 features) | All 7 | `shap/shap_dependence_{model}_{feat}.png` |
| Waterfall (per country) | Best model | `shap/shap_waterfall_{country}_{model}.png` |
| Force plots (HTML) | Best model | `shap/shap_force_{country}_{model}.html` |
| Interaction effects | Best tree model | `shap/shap_interaction_{model}_{f1}_x_{f2}.png` |
| Scenario comparison (SSP1/4/5) | Best model | `shap/shap_scenario_comparison_{country}_{model}.png` |
| GAM smooth functions | GAM only | `gam_partial_dependence/pd_{feature}.png` |
| Cross-model importance ranking | All 7 | `shap_feature_importance_comparison.csv` |

**Primary threshold**: $3/day (most policy-relevant; supervisor reference threshold).


## 0. Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings
warnings.filterwarnings("ignore")

from explainability import (
    run_full_shap_analysis,
    load_training_artefacts,
    load_test_metadata,
    load_model_bundle,
    get_explainer,
    compute_shap_values,
    plot_summary,
    plot_importance_bar,
    plot_dependence,
    plot_waterfall,
    save_force_plots,
    plot_interactions,
    plot_scenario_shap_comparison,
    plot_gam_partial_dependence,
    build_importance_comparison,
    print_consensus_summary,
    FEATURE_LABELS,
    INTERESTING_COUNTRIES,
    PRIMARY_THRESHOLD,
    SHAP_DIR,
    GAM_PD_DIR,
)
from config import DATA_FINAL_DIR, DATA_PROCESSED_DIR, MODELS_DIR, OUTPUTS_DIR
from model_pipeline import MODEL_NAMES

shap.initjs()
print("SHAP version:", shap.__version__)
print("Output dirs:")
print(f"  SHAP plots: {SHAP_DIR}")
print(f"  GAM PD:     {GAM_PD_DIR}")

## 1. Load training artefacts

In [ ]:
arts = load_training_artefacts(DATA_FINAL_DIR)
X_train    = arts["X_train"].values.astype(float)
X_test     = arts["X_test"].values.astype(float)
y_test     = arts["y_test"]["poverty_3"].values.astype(float)
feat_names = arts["feature_names"]

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"Features: {feat_names}")

# Identify best model from previous run
comparison_path = OUTPUTS_DIR / "model_comparison_approach_a.csv"
if comparison_path.exists():
    comp = pd.read_csv(comparison_path)
    best_model_name = comp.sort_values("rmse").iloc[0]["model_name"]
    print(f"\nBest model (lowest RMSE, $3/day): {best_model_name}")
    print(comp[["model_name","rmse","r2"]].sort_values("rmse").to_string(index=False))
else:
    best_model_name = "xgboost_cpu"
    print(f"Comparison CSV not found — defaulting to {best_model_name}")

## 2. Test set metadata (country labels)

In [ ]:
print("Reconstructing country labels for test rows (may take ~60s on first run)…")
try:
    test_meta = load_test_metadata()
    print(f"Test metadata: {len(test_meta)} rows, {test_meta['country_name'].nunique()} countries")
    print(test_meta.head(10).to_string())
except Exception as e:
    print(f"WARNING: {e}")
    test_meta = None

## 3. Run complete SHAP pipeline (all 7 models)

In [ ]:
# This is the main call — runs all SHAP analyses for all 7 models.
#
# Runtime estimates (approximate, hardware-dependent):
#   Tree models (XGBoost, LightGBM, RF) : ~10–60 s each
#   Ridge (LinearExplainer)             : ~5 s
#   GAM  (KernelExplainer)              : ~5–15 min
#   MLP  (KernelExplainer)              : ~5–15 min
#
# Set kernel_subsample to reduce MLP/GAM time (default 200 rows).

result = run_full_shap_analysis(
    final_dir           = DATA_FINAL_DIR,
    models_dir          = MODELS_DIR,
    outputs_dir         = OUTPUTS_DIR,
    threshold           = "$3",
    best_model_override = None,   # set to "xgboost_cpu" to force a specific model
    kernel_subsample    = 200,
    scenario_countries  = ["Nigeria", "India", "Germany"],
    waterfall_countries = None,   # None = use INTERESTING_COUNTRIES default
)

## 4. Cross-model importance comparison

In [ ]:
imp_df = pd.read_csv(OUTPUTS_DIR / "shap_feature_importance_comparison.csv", index_col=0)
model_cols = [m for m in MODEL_NAMES if m in imp_df.columns]

print("Feature importance (mean |SHAP|) across all models:")
display_cols = model_cols + ["mean_across_models", "rank_mean"]
print(imp_df[[c for c in display_cols if c in imp_df.columns]].round(4).to_string())

## 5. Display key plots inline

### 5a. Best model — SHAP summary (beeswarm)

In [ ]:
from IPython.display import Image, display
p = SHAP_DIR / f"shap_summary_{best_model_name}.png"
if p.exists():
    display(Image(str(p)))
else:
    print(f"File not found: {p}")

### 5b. Best model — SHAP importance bar

In [ ]:
p = SHAP_DIR / f"shap_importance_{best_model_name}.png"
if p.exists():
    display(Image(str(p)))
else:
    print(f"File not found: {p}")

### 5c. Best model — top-3 dependence plots

In [ ]:
# Show all dependence plots for the best model
dep_files = sorted(SHAP_DIR.glob(f"shap_dependence_{best_model_name}_*.png"))
print(f"Dependence plots for {best_model_name}: {[f.name for f in dep_files]}")
for p in dep_files:
    display(Image(str(p)))

### 5d. Waterfall plots — rich, middle, poor country

In [ ]:
wf_files = sorted(SHAP_DIR.glob(f"shap_waterfall_*_{best_model_name}.png"))
print(f"Waterfall plots: {[f.name for f in wf_files]}")
for p in wf_files:
    print(f"\n{p.stem}")
    display(Image(str(p)))

### 5e. Force plots (HTML)

In [ ]:
force_files = sorted(SHAP_DIR.glob(f"shap_force_*_{best_model_name}.html"))
print("Force plot files (open in browser):")
for p in force_files:
    print(f"  {p}")

### 5f. Interaction effects

In [ ]:
interact_files = sorted(SHAP_DIR.glob(f"shap_interaction_{best_model_name}_*.png"))
print(f"Interaction plots: {[f.name for f in interact_files]}")
for p in interact_files:
    display(Image(str(p)))

### 5g. Scenario comparison — SSP1 vs SSP4 vs SSP5

In [ ]:
scenario_files = sorted(SHAP_DIR.glob(f"shap_scenario_comparison_*_{best_model_name}.png"))
print(f"Scenario comparison plots: {[f.name for f in scenario_files]}")
for p in scenario_files:
    print(f"\n{p.stem.replace('shap_scenario_comparison_','').replace(f'_{best_model_name}','')}")
    display(Image(str(p)))

### 5h. GAM partial dependence (unique advantage)

In [ ]:
# GAM smooth functions — exact partial dependence, not SHAP approximations
gam_pd_files = sorted(GAM_PD_DIR.glob("*.png"))
print(f"GAM partial dependence plots: {len(gam_pd_files)}")

# Show the overview grid first
grid_file = GAM_PD_DIR / "pd_all_features_grid.png"
if grid_file.exists():
    print("Overview grid:")
    display(Image(str(grid_file)))

## 6. Manual deep-dives

### 6a. Recompute SHAP for a specific model interactively

In [ ]:
# Change this to any model you want to inspect
INSPECT_MODEL = best_model_name

bundle    = load_model_bundle(INSPECT_MODEL, "$3")
model_obj = bundle["model"]
expl, used_idx = compute_shap_values(
    INSPECT_MODEL, model_obj, X_test, X_train,
    n_background=100, kernel_subsample=300,
)
print(f"SHAP values shape: {expl.values.shape}")
print(f"Base value (expected prediction): {np.mean(expl.base_values):.3f}%")

### 6b. Feature importance for a single observation

In [ ]:
# Pick a specific row (e.g. first row of the subsampled test set)
row_idx = 0
sv_row  = expl.values[row_idx]
feat_labels = [FEATURE_LABELS.get(f, f) for f in feat_names]

order = np.argsort(np.abs(sv_row))[::-1]
print(f"SHAP contributions for test row {row_idx}:")
print(f"  Base value: {expl.base_values[row_idx]:.3f}")
print(f"  Prediction: {expl.base_values[row_idx] + sv_row.sum():.3f}%")
print()
for i in order:
    print(f"  {feat_labels[i]:<40}  {sv_row[i]:+.4f}  "
          f"(feature val: {expl.data[row_idx, i]:.4f})")

### 6c. Ridge SHAP vs model coefficients

In [ ]:
# Ridge is fully interpretable: SHAP = coefficient × (feature - background mean)
# Verify this matches the Ridge coefficients directly
try:
    ridge_bundle = load_model_bundle("ridge", "$3")
    ridge_model  = ridge_bundle["model"]

    ridge_expl, _ = compute_shap_values(
        "ridge", ridge_model, X_test, X_train,
        n_background=100,
    )

    coefs = ridge_model.coef_
    print("Ridge coefficients vs mean |SHAP|:")
    mean_abs_shap = np.abs(ridge_expl.values).mean(axis=0)
    for feat, coef, shap_val in zip(feat_names, coefs, mean_abs_shap):
        label = FEATURE_LABELS.get(feat, feat)[:38]
        print(f"  {label:<40}  coef={coef:+.4f}  mean|SHAP|={shap_val:.4f}")
except Exception as e:
    print(f"Skipped: {e}")

### 6d. GAM summary statistics

In [ ]:
try:
    gam_bundle = load_model_bundle("gam", "$3")
    gam_model  = gam_bundle["model"]
    print(gam_model.summary())
except Exception as e:
    print(f"Skipped: {e}")

## 7. Cross-model agreement analysis

In [ ]:
# Which features have consistent rankings across ALL 7 models?
model_cols = [m for m in MODEL_NAMES if m in imp_df.columns]
ranks = imp_df[model_cols].rank(ascending=False)

print("Feature ranking across models (1 = most important):")
print(f"{'Feature':<40}  " + "  ".join(f"{m[:12]:>12}" for m in model_cols) + "  Spread")
print("-" * 120)
for feat in imp_df.index:
    row = ranks.loc[feat, model_cols]
    spread = row.max() - row.min()
    label = FEATURE_LABELS.get(feat, feat)[:38]
    rank_strs = "  ".join(f"{int(v):>12}" for v in row.values)
    print(f"{label:<40}  {rank_strs}  {int(spread):>6}")

print("\nConsensus: low spread = all models agree on this feature's importance")

## 8. Verify all output files

In [ ]:
print("=== SHAP outputs ===")
shap_files = sorted(SHAP_DIR.rglob("*.png")) + sorted(SHAP_DIR.rglob("*.html"))
print(f"Total: {len(shap_files)}")
for p in shap_files:
    print(f"  {p.name:<60}  {p.stat().st_size:>8,} bytes")

print(f"\n=== GAM partial dependence ===")
gam_files = sorted(GAM_PD_DIR.rglob("*.png"))
print(f"Total: {len(gam_files)}")
for p in gam_files:
    print(f"  {p.name}")

print(f"\n=== Summary CSV ===")
csv_path = OUTPUTS_DIR / "shap_feature_importance_comparison.csv"
if csv_path.exists():
    print(f"  {csv_path.name}  ({csv_path.stat().st_size:,} bytes)")

## 9. Key findings for report

Use the cells below to draft takeaways based on actual results.
Placeholders are provided for the main findings.

In [ ]:
print_consensus_summary(imp_df)

## 10. Report discussion template

Paste actual numbers once results are available.

---

### Feature Importance Consensus

Across all 7 models, **[FEATURE_1]** and **[FEATURE_2]** consistently rank as the most
important predictors of poverty headcount at the $3/day threshold.
This is consistent with economic theory: [interpretation].

### Non-linear Effects (GAM advantage)

The GAM partial dependence plots reveal that the relationship between
`log(GDP per capita)` and poverty is **[linear / concave / convex]**, suggesting
[interpretation]. The sigmoid-like shape in the HDI curve indicates a tipping
point around HDI ≈ [value].

### Directional Effects (SHAP beeswarm)

- **Higher GDP per capita** → SHAP pushes prediction **down** (reduces poverty). Expected.
- **Higher Gini coefficient** → SHAP pushes prediction **up** (more inequality = more poverty). 
- **Higher Control of Corruption** → SHAP pushes prediction **[up/down]**.
- **Employment in agriculture** → positive effect in low-income countries (subsistence),
  negative in high-income countries (structural transformation complete).

### Scenario Comparison (Nigeria example)

Under SSP1 (sustainability), the model predicts [X]% poverty by 2050 vs SSP4 [Y]% and SSP5 [Z]%.
The SHAP scenario comparison shows that the GDP per capita term accounts for [N]pp of
the SSP1–SSP5 gap, while the Gini term adds [M]pp.

### Model Agreement

Models [A, B, C] agree on the top-3 features. Models [D, E] disagree on
[feature] — tree models give it higher importance, suggesting a non-linear
threshold effect that linear Ridge cannot capture.

---

*Fill in once notebook has been run.*


## 11. Summary

| Plot type | Count | Location |
|-----------|-------|----------|
| SHAP beeswarm | 7 | `outputs/shap/` |
| SHAP importance bar | 7 | `outputs/shap/` |
| SHAP dependence (top-3) | ≤21 | `outputs/shap/` |
| Waterfall (3 countries) | ≤3 | `outputs/shap/` |
| Force plots (HTML) | ≤3 | `outputs/shap/` |
| Interaction effects | ≤4 | `outputs/shap/` |
| Scenario comparison | ≤3×3 | `outputs/shap/` |
| GAM partial dependence | 13+1 | `outputs/gam_partial_dependence/` |
| Importance comparison CSV | 1 | `outputs/` |
| Importance heatmap | 1 | `outputs/shap/` |
